# 🚗 Car Type Classifier — Notebook 3 : Classification Zero-Shot & Évaluation

**Modèle** : `facebook/bart-large-mnli`  
**Approche** : Zero-shot classification (aucun entraînement requis)  
**Labels** : `['family car', 'sport car']`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os
from tqdm.notebook import tqdm
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)
from transformers import pipeline

sys.path.append('../src')
sns.set_theme(style='whitegrid')
print('✅ Librairies chargées')

## 1. Chargement du dataset prétraité

In [ ]:
preprocessed_path = '../results/dataset_preprocessed.csv'

if os.path.exists(preprocessed_path):
    df = pd.read_csv(preprocessed_path)
    print(f'✅ Dataset chargé : {len(df)} voitures')
else:
    print('⚠️ Dataset prétraité non trouvé — exécution depuis le début')
    from classifier import load_and_clean_data, build_text_description
    df = load_and_clean_data('../data/')
    df['text_description'] = df.apply(build_text_description, axis=1)

print('\nAperçu des descriptions :')
df[['text_description']].head(5)

## 2. Chargement du modèle Zero-Shot

In [ ]:
print('🤖 Chargement du modèle facebook/bart-large-mnli...')
print('   (Premier chargement : ~1.6 GB, peut prendre quelques minutes)')

classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=-1  # CPU ; mettre 0 pour GPU si disponible
)
print('✅ Modèle chargé !')

## 3. Test rapide sur quelques exemples

In [ ]:
LABELS = ['family car', 'sport car']

test_examples = [
    'Toyota Verso, 7 seats, 120 HP, top speed 185 km/h, Hybrid',
    'Ferrari 488 GTB, 2 seats, 670 HP, top speed 330 km/h, Petrol',
    'Volkswagen Sharan, 7 seats, 150 HP, top speed 200 km/h, Diesel',
    'Porsche 911 Carrera, 2 seats, 450 HP, top speed 293 km/h, Petrol',
]

print('🧪 Tests sur quelques exemples :\n')
for text in test_examples:
    result = classifier(text, candidate_labels=LABELS)
    pred = result['labels'][0]
    score = result['scores'][0]
    icon = '🏁' if pred == 'sport car' else '👨‍👩‍👧'
    print(f'{icon} [{pred.upper()} — {score:.1%}] {text}')

## 4. Classification du dataset complet

In [ ]:
predictions = []
scores_family = []
scores_sport = []

for text in tqdm(df['text_description'], desc='Classification en cours'):
    result = classifier(text, candidate_labels=LABELS)
    score_dict = dict(zip(result['labels'], result['scores']))
    predictions.append(result['labels'][0])
    scores_family.append(score_dict.get('family car', 0))
    scores_sport.append(score_dict.get('sport car', 0))

df['predicted_label'] = predictions
df['score_family'] = scores_family
df['score_sport'] = scores_sport
df['predicted_class'] = df['predicted_label'].map({'family car': 'family', 'sport car': 'sport'})

print('✅ Classification terminée !')
print(f"\nRépartition des prédictions :\n{df['predicted_class'].value_counts()}")

## 5. Évaluation des performances

In [ ]:
if 'true_label' in df.columns:
    y_true = df['true_label']
    y_pred = df['predicted_class']
    
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, pos_label='sport', zero_division=0)
    rec  = recall_score(y_true, y_pred, pos_label='sport', zero_division=0)
    f1   = f1_score(y_true, y_pred, pos_label='sport', zero_division=0)

    print('='*45)
    print('📊 MÉTRIQUES D\'ÉVALUATION')
    print('='*45)
    print(f'  Accuracy  : {acc:.2%}')
    print(f'  Precision : {prec:.2%}')
    print(f'  Recall    : {rec:.2%}')
    print(f'  F1-score  : {f1:.2%}')
    print('\nRapport de classification :')
    print(classification_report(y_true, y_pred, zero_division=0))
else:
    print('⚠️ Pas de colonne true_label : évaluation ignorée.')

## 6. Visualisations

In [ ]:
os.makedirs('../results', exist_ok=True)

if 'true_label' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Matrice de confusion
    cm = confusion_matrix(df['true_label'], df['predicted_class'], labels=['family', 'sport'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Familiale', 'Sportive'],
                yticklabels=['Familiale', 'Sportive'], ax=axes[0])
    axes[0].set_title('Matrice de confusion', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Prédiction')
    axes[0].set_ylabel('Réalité')

    # Bar chart métriques
    metric_values = [acc, prec, rec, f1]
    metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-score']
    colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
    bars = axes[1].bar(metric_names, metric_values, color=colors, edgecolor='white', linewidth=1.5)
    axes[1].set_ylim(0, 1.15)
    axes[1].set_title('Métriques d\'évaluation', fontsize=13, fontweight='bold')
    for bar, val in zip(bars, metric_values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.2%}', ha='center', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../results/evaluation_metrics.png', dpi=150)
    plt.show()

# Distribution des scores de confiance
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df['score_family'], bins=20, alpha=0.7, label='Score Familiale', color='#4C72B0')
ax.hist(df['score_sport'],  bins=20, alpha=0.7, label='Score Sportive',  color='#C44E52')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Seuil 0.5')
ax.set_title('Distribution des scores de confiance (zero-shot)', fontsize=13, fontweight='bold')
ax.set_xlabel('Score de confiance')
ax.set_ylabel('Nombre de voitures')
ax.legend()
plt.tight_layout()
plt.savefig('../results/confidence_scores.png', dpi=150)
plt.show()
print('✅ Graphiques sauvegardés dans results/')

## 7. Analyse des erreurs

In [ ]:
if 'true_label' in df.columns:
    errors = df[df['true_label'] != df['predicted_class']]
    print(f'❌ Erreurs de classification : {len(errors)} / {len(df)} ({len(errors)/len(df):.1%})')
    print('\nVoitures mal classifiées :')
    display_cols = ['text_description', 'true_label', 'predicted_class', 'score_family', 'score_sport']
    print(errors[display_cols].to_string(index=False))
else:
    print('Pas de vraies étiquettes pour analyser les erreurs.')

## 8. Sauvegarde des résultats

In [ ]:
df.to_csv('../results/predictions.csv', index=False)
print(f'✅ Prédictions sauvegardées : results/predictions.csv')
df[['text_description', 'predicted_label', 'score_family', 'score_sport']].head(10)

## ✅ Conclusion

### Ce que nous avons fait :
1. **Chargé** deux datasets de voitures depuis Kaggle
2. **Nettoyé et transformé** les données en descriptions textuelles
3. **Appliqué** `facebook/bart-large-mnli` en classification zero-shot
4. **Évalué** avec Accuracy, Precision, Recall, F1-score et matrice de confusion

### Analyse critique :
- Le modèle zero-shot fonctionne sans aucun entraînement sur des données voitures
- Les erreurs surviennent principalement sur des voitures hybrides (ex. SUV sportif ou berline rapide)
- L'attribut **nombre de places** est le signal le plus discriminant

### Améliorations possibles :
- Few-shot prompting avec des exemples dans le texte
- Fine-tuning sur un dataset annoté voitures
- Ajout de plus d'attributs (dimensions, poids, type de carrosserie)